In [1]:
# 📦 Install all required packages
!pip install langchain langchain-community langchain-chroma chromadb sentence-transformers gradio transformers accelerate torch -q

print("✅ Dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.

In [2]:
# ⚙️ Configuration & Safety Settings

CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die",
    "hurt myself", "self harm", "cutting", "overdose",
    "kill someone", "hurt them", "no hope", "everything is hopeless"
]

SYSTEM_PROMPT = """
You are a supportive wellness companion, NOT a licensed therapist.
1. You cannot diagnose conditions or prescribe medication.
2. If a user asks for medical advice, suggest consulting a professional.
3. Always respond with empathy.
4. If the answer is not in your context, say you don't know and suggest professional help.
5. END EVERY RESPONSE with: "(Reminder: I am an AI tool, not a human therapist.)"
"""

print("✅ Configuration loaded!")

✅ Configuration loaded!


In [3]:
from langchain_core.documents import Document

# 🛡️ Safety Layer Class
class SafetyLayer:
    def check_input(self, user_input: str) -> dict:
        input_lower = user_input.lower()
        for keyword in CRISIS_KEYWORDS:
            if keyword in input_lower:
                return {
                    "is_safe": False,
                    "response": (
                        "⚠️ **Important:** It sounds like you're going through something really difficult. "
                        "I am an AI and cannot provide emergency support. \n\n"
                        "Please reach out for human help immediately: **Call/Text 988** (US) or go to your nearest ER.\n\n"
                        "You are not alone."
                    )
                }
        return {"is_safe": True, "response": None}

# 📚 FAQ Data (Clinically Reviewed Content)
faq_data = [
    {
        "question": "How can I calm down during a panic attack?",
        "answer": "Try the 5-4-3-2-1 grounding technique: Acknowledge 5 things you see, 4 you can touch, 3 you hear, 2 you can smell, and 1 you can taste. Focus on breathing: inhale 4s, hold 4s, exhale 4s. [Source: ADAA]",
        "category": "coping_skills"
    },
    {
        "question": "What is burnout?",
        "answer": "Burnout is emotional, physical, and mental exhaustion caused by excessive stress. It makes you feel overwhelmed and drained. [Source: Mayo Clinic]",
        "category": "psychoeducation"
    },
    {
        "question": "How do I find a therapist?",
        "answer": "Search directories like Psychology Today, contact your insurance provider, or ask your primary care physician. Many communities offer sliding-scale clinics. [Source: APA]",
        "category": "resources"
    },
    {
        "question": "What are some good sleep hygiene tips?",
        "answer": "Maintain a consistent sleep schedule, avoid screens 1 hour before bed, keep your room cool and dark, and avoid caffeine after 2pm. [Source: Sleep Foundation]",
        "category": "wellness"
    },
    {
        "question": "How can I manage stress?",
        "answer": "Try regular exercise, mindfulness meditation, deep breathing, talking to friends, and setting healthy boundaries. [Source: APA]",
        "category": "coping_skills"
    }
]

# Convert to LangChain Documents
documents = [
    Document(
        page_content=f"Question: {item['question']}\nAnswer: {item['answer']}",
        metadata={"category": item["category"]}
    )
    for item in faq_data
]

print(f"✅ Loaded {len(documents)} FAQ documents!")

✅ Loaded 5 FAQ documents!


In [4]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 🧠 Load FREE local embedding model
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

# Create Vector Store
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("✅ Vector Store created!")

Loading embedding model...


/tmp/ipykernel_3480/697512287.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector Store created!


In [6]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import torch

# 🤖 Use a small causal LM that works with "text-generation" task
# TinyLlama is small (~2GB) and works well for FAQ tasks
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading model: {model_name}... (this may take 2-3 minutes on first run)")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True
)

# Create HuggingFace pipeline with CORRECT task name
pipe = pipeline(
    "text-generation",  # ✅ Updated task name (was "text2text-generation")
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.1,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,  # Required for batching
)

# Wrap with LangChain
llm = HuggingFacePipeline(pipeline=pipe)

# Format function for RAG
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context: {context}\n\nUser Question: {input}"),
])

# Build RAG chain
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Local LLM loaded and ready!")

Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0... (this may take 2-3 minutes on first run)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'pad_token_id', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Local LLM loaded and ready!


/tmp/ipykernel_3480/3811519300.py:35: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [7]:
import gradio as gr

safety_layer = SafetyLayer()

def bot_response(message, history):
    # 1. Safety Check FIRST (Critical for mental health)
    safety_check = safety_layer.check_input(message)
    if not safety_check["is_safe"]:
        return safety_check["response"]

    # 2. RAG Query
    try:
        response = rag_chain.invoke(message)
        return response
    except Exception as e:
        if "rate limit" in str(e).lower():
            return "⏳ I'm temporarily busy. Please wait 30 seconds and try again. For urgent support, call 988."
        return f"Error: {str(e)}"

# Create Gradio Interface
demo = gr.ChatInterface(
    fn=bot_response,
    title="🧠 Wellness Companion (FREE Demo)",
    description="⚠️ **NOT A THERAPIST.** For emergencies: Call/Text **988**",
    theme="soft",
    examples=["How can I calm down?", "What is burnout?", "How do I find help?", "Sleep tips?"]
)

print("🚀 Launching chat interface...")
demo.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


🚀 Launching chat interface...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a3d8e5b533b1f7e8a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
# 📦 Install required packages for Groq + LangChain + Chroma
!pip install langchain langchain-community langchain-chroma chromadb sentence-transformers gradio groq langchain-groq -q

print("✅ Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00
✅ Dependencies installed!


In [15]:
# Run this in Cell 1 instead:
!pip install langchain langchain-community langchain-chroma chromadb sentence-transformers gradio==4.44.0 groq langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires websockets>=14.0, but you have websockets 12.0 which is incompatible.
google-genai 1.68.0 requires websockets<17.0,>=13.0.0, but you have websockets 12.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
google-adk 1.28.0 requires websockets<16.0.0,

In [16]:
# ⚙️ Configuration
import os
from getpass import getpass

# 🔒 Get Groq API Key securely
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("🔑 Enter your Groq API Key (starts with gsk_): ")

# 🛡️ Crisis Detection Keywords
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die",
    "hurt myself", "self harm", "cutting", "overdose",
    "kill someone", "hurt them", "no hope", "everything is hopeless",
    "i'm done", "can't go on", "end it all"
]

# 🤖 System Prompt with Clear Boundaries
SYSTEM_PROMPT = """
You are a supportive wellness companion, NOT a licensed therapist, doctor, or crisis counselor.

YOUR BOUNDARIES:
1. You cannot diagnose mental health conditions or prescribe medication.
2. You cannot provide therapy sessions or medical advice.
3. If a user asks for diagnosis/medication, gently suggest consulting a professional.
4. Always respond with empathy, warmth, and validation.
5. Keep responses concise (2-4 sentences) and actionable.

YOUR GOAL:
Provide general wellness information, coping strategies, and psychoeducation based ONLY on the provided context.

IF YOU DON'T KNOW:
If the answer is not in the context, say: "I don't have specific information on that, but it sounds important. Please consider discussing this with a mental health professional."

ALWAYS INCLUDE DISCLAIMER:
End every response with: "(Reminder: I am an AI wellness tool, not a human therapist.)"
"""

print("✅ Configuration loaded!")

✅ Configuration loaded!


In [17]:
from langchain_core.documents import Document

# 🛡️ Safety Layer Class
class SafetyLayer:
    def check_input(self, user_input: str) -> dict:
        input_lower = user_input.lower()
        for keyword in CRISIS_KEYWORDS:
            if keyword in input_lower:
                return {
                    "is_safe": False,
                    "response": (
                        "⚠️ **Important:** It sounds like you're going through something really difficult. "
                        "I am an AI and cannot provide emergency support. \n\n"
                        "🆘 **Please reach out for human help immediately:**\n"
                        "• 🇺🇸 US: Call/Text **988** or chat at [988lifeline.org](https://988lifeline.org)\n"
                        "• 🇬🇧 UK: Call **116 123** (Samaritans)\n"
                        "• 🌍 Global: Go to your nearest emergency room\n\n"
                        "You are not alone, and help is available right now. 💙"
                    )
                }
        return {"is_safe": True, "response": None}

# 📚 FAQ Data (Clinically-Inspired Content)
faq_data = [
    {
        "question": "How can I calm down during a panic attack?",
        "answer": "Try the 5-4-3-2-1 grounding technique: Name 5 things you see, 4 you can touch, 3 you hear, 2 you can smell, 1 you can taste. Then breathe: inhale 4 seconds, hold 4, exhale 6. Repeat 3-4 times. This activates your body's relaxation response. [Source: Anxiety & Depression Association of America]",
        "category": "coping_skills"
    },
    {
        "question": "What is burnout?",
        "answer": "Burnout is a state of emotional, physical, and mental exhaustion from prolonged stress. Signs include feeling drained, cynical, and ineffective. Recovery involves rest, boundaries, and support. [Source: Mayo Clinic]",
        "category": "psychoeducation"
    },
    {
        "question": "How do I find a therapist?",
        "answer": "Try these steps: 1) Search Psychology Today's therapist directory, 2) Contact your insurance for in-network providers, 3) Ask your primary care doctor for referrals, 4) Look into community mental health centers for sliding-scale options. [Source: American Psychological Association]",
        "category": "resources"
    },
    {
        "question": "What are good sleep hygiene tips?",
        "answer": "Keep a consistent sleep schedule (even weekends), avoid screens 1 hour before bed, keep your room cool/dark/quiet, limit caffeine after 2pm, and create a relaxing pre-sleep routine. [Source: Sleep Foundation]",
        "category": "wellness"
    },
    {
        "question": "How can I manage daily stress?",
        "answer": "Try: 1) 5-minute breathing exercises, 2) Short walks outside, 3) Writing down worries, 4) Talking to a trusted friend, 5) Setting small, achievable goals. Even tiny steps help. [Source: APA]",
        "category": "coping_skills"
    },
    {
        "question": "Is it normal to feel anxious?",
        "answer": "Yes, anxiety is a normal human response to stress. It becomes a concern when it's frequent, intense, or interferes with daily life. If worry feels overwhelming, talking to a professional can help. [Source: NIMH]",
        "category": "psychoeducation"
    }
]

# Convert to LangChain Documents
documents = [
    Document(
        page_content=f"Question: {item['question']}\nAnswer: {item['answer']}",
        metadata={"category": item["category"], "source": item.get("source", "Curated")}
    )
    for item in faq_data
]

print(f"✅ Loaded {len(documents)} clinically-reviewed FAQ entries!")

✅ Loaded 6 clinically-reviewed FAQ entries!


In [18]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 🧠 Load FREE local embedding model (runs in Colab)
print("Loading embedding model (~500MB download)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

# Create Vector Store
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})  # Return top 2 relevant FAQs

print("✅ Vector Store created with Chroma!")

Loading embedding model (~500MB download)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector Store created with Chroma!


In [19]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 🚀 Initialize Groq LLM
llm = ChatGroq(
    model="llama3-8b-8192",  # Fast, high-quality, free tier
    # Alternative models:
    # - "llama3-70b-8192" (more powerful, slightly slower)
    # - "mixtral-8x7b-32768" (excellent for reasoning)
    temperature=0.1,          # Low = more consistent/factual
    max_tokens=300,           # Keep responses concise
    timeout=30                # Prevent hanging
)

# 📝 Format retrieved documents for context
def format_docs(docs):
    return "\n\n".join([f"• {doc.page_content}" for doc in docs])

# 💬 Create the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Relevant FAQ Context:\n{context}\n\nUser Question: {input}"),
])

# 🔗 Build the RAG chain
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Groq LLM connected! Using llama3-8b-8192")
print(f"💡 Free tier: ~30 requests/minute")

✅ Groq LLM connected! Using llama3-8b-8192
💡 Free tier: ~30 requests/minute


In [22]:
import gradio as gr

safety_layer = SafetyLayer()

def bot_response(message, history):
    """
    Main chat handler with safety + RAG
    """
    # 🛡️ STEP 1: Crisis detection (ALWAYS FIRST)
    safety_check = safety_layer.check_input(message)
    if not safety_check["is_safe"]:
        return safety_check["response"]

    # 🔍 STEP 2: RAG query with error handling
    try:
        response = rag_chain.invoke(message)
        return response
    except Exception as e:
        error_msg = str(e).lower()
        if "rate limit" in error_msg or "429" in error_msg:
            return "⏳ I'm temporarily at capacity. Please wait ~30 seconds and try again. For urgent support, call 988."
        elif "api key" in error_msg:
            return "🔑 API key issue. Please restart and re-enter your Groq key."
        else:
            return "🤔 I'm having trouble processing that. Please try rephrasing, or contact a professional for urgent matters."

# 🎨 Create Gradio Interface (FIXED PARAMETERS)
demo = gr.ChatInterface(
    fn=bot_response,
    title="🧠 Wellness Companion",
    description="⚠️ **NOT A THERAPIST** | For emergencies: 🇺🇸 Call/Text **988** | 🌍 [Find global resources](https://findahelpline.com)",
    examples=[
        "How can I calm down during anxiety?",
        "What is burnout?",
        "How do I find a therapist?",
        "Sleep tips for stress?",
        "Is it normal to feel this way?"
    ],
    theme="soft"
)

print("🚀 Launching chat interface...")
print("💡 Click the public URL below to chat in your browser")
demo.launch(share=True)  # share=True creates a public link (expires in 72h)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


🚀 Launching chat interface...
💡 Click the public URL below to chat in your browser
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()


/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:93: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  await asyncio.wait_for(


* Running on public URL: https://8b1d5f0b206b9dad9a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
